# Oiled — Stage 3: Triage & Evaluation (Part 3)

**Goal:** Load the trained U-Net (`seg_best.pt`), run pixel-level inference on **all 450 Part-3 scenes** (Oil / Lookalike / No oil), derive an optimal detection threshold from the validation precision-recall curve, then report scene-level metrics broken down by category.

**Inputs required in Kaggle Output (from the Stage 2 session):**
- `/kaggle/working/02_Test_images_and_ground_truth.7z` — the raw dataset (already present)
- `seg_best.pt` — upload as a Kaggle dataset or re-upload as a notebook input

**Outputs:**
- `triage_report.json` — threshold + per-category scene-level metrics
- `pr_curve.png` — precision-recall curve with operating point
- `confusion_matrix.png` — scene-level confusion matrix (Oil vs Lookalike vs Clean)
- `scene_results.csv` — per-scene inference results

In [ ]:
import torch, platform
print('=== Oiled — Stage 3: Triage & Evaluation ===')
print(f'Python:  {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPUs:    {torch.cuda.device_count()}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected! Enable GPU in Kaggle sidebar.')

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | VRAM: {p.total_memory // 1024**2} MiB')

DEVICE = torch.device('cuda:0')

In [ ]:
!apt-get install -y -qq p7zip-full
!pip install -q timm==0.9.7 segmentation-models-pytorch==0.3.4 rasterio==1.4.3 albumentations==1.4.23 scikit-learn
print('✓ Dependencies installed')

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import rasterio
import torch
import torch.nn as nn
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import segmentation_models_pytorch as smp
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────
ARCHIVE   = Path('/kaggle/working/02_Test_images_and_ground_truth.7z')
DATA_ROOT = Path('/tmp/data/part3')
WORK_DIR  = Path('/kaggle/working')

# seg_best.pt location — try several candidate paths
CKPT_CANDIDATES = [
    WORK_DIR / 'seg_best.pt',
    Path('/kaggle/input/oiled-seg-v01/seg_best.pt'),   # if uploaded as dataset
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), None)
if CKPT_PATH is None:
    raise FileNotFoundError(
        'seg_best.pt not found. Upload it as a Kaggle dataset input '
        'or place it in /kaggle/working/'
    )
print(f'✓ Checkpoint: {CKPT_PATH}')

# ── hyperparams (must match Stage 2 training exactly) ──────────────────────
TILE_SIZE   = 512
STRIDE      = 384
CLIP_MIN    = -40.0
CLIP_MAX    =  10.0
BATCH_TILES =  8     # tiles per inference batch

print('✓ Config OK')

In [ ]:
# ── Extract dataset (skip if already done) ─────────────────────────────────
if not (DATA_ROOT / 'Images').exists():
    print('Extracting archive … (may take 3-5 min)')
    t0 = time.time()
    os.makedirs(DATA_ROOT, exist_ok=True)
    ret = os.system(f'7z x "{ARCHIVE}" -o"{DATA_ROOT}" -y -bd > /tmp/7z.log 2>&1')
    print(f'Exit code {ret} — {time.time()-t0:.0f}s')
    if ret != 0:
        os.system('tail -20 /tmp/7z.log')
        raise RuntimeError('Extraction failed')
else:
    print('✓ Archive already extracted')

# ── Discover images & masks ────────────────────────────────────────────────
IMG_DIRS  = list((DATA_ROOT / 'Images').rglob('*.tif'))
MASK_DICT = {}
for m in (DATA_ROOT / 'Mask').rglob('*.tif'):
    key = m.stem.replace('_segmentation', '').strip('_')
    MASK_DICT[key] = m

def get_category(path: Path) -> str:
    s = str(path).lower()
    if 'lookalike' in s: return 'lookalike'
    if 'no' in s:        return 'clean'
    return 'oil'

records = []
for img_path in IMG_DIRS:
    key = img_path.stem
    if key in MASK_DICT:
        records.append({'scene_id': key,
                        'img_path': img_path,
                        'mask_path': MASK_DICT[key],
                        'category': get_category(img_path)})

df = pd.DataFrame(records)
print(f'\nPaired scenes: {len(df)}')
print(df['category'].value_counts().to_string())

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=2,
    classes=1,
    activation=None,
)

state = torch.load(CKPT_PATH, map_location='cpu')
# Handle DataParallel prefix if present
if any(k.startswith('module.') for k in state.keys()):
    state = {k.replace('module.', '', 1): v for k, v in state.items()}
model.load_state_dict(state)
model = model.to(DEVICE).eval()

print(f'✓ Model loaded from {CKPT_PATH.name}')
n_params = sum(p.numel() for p in model.parameters())
print(f'  Parameters: {n_params/1e6:.1f} M')

In [ ]:
# ── Inference helpers ──────────────────────────────────────────────────────

def normalize_sar(arr: np.ndarray) -> np.ndarray:
    """Clip and normalise SAR Sigma0 dB → [0, 1]."""
    arr = np.clip(arr, CLIP_MIN, CLIP_MAX)
    return (arr - CLIP_MIN) / (CLIP_MAX - CLIP_MIN)


def read_scene(img_path: Path):
    """Return (H, W, 2) float32 normalised array."""
    with rasterio.open(img_path) as src:
        bands = src.read().astype(np.float32)   # (C, H, W)
    # Some scenes have >2 bands; take first two (VV, VH)
    bands = bands[:2]
    bands = normalize_sar(bands)
    return bands  # (2, H, W)


def read_mask(mask_path: Path) -> np.ndarray:
    """Return binary (H, W) uint8 mask."""
    with rasterio.open(mask_path) as src:
        m = src.read(1).astype(np.float32)
    return (m > 0.5).astype(np.uint8)


def inference_scene(img: np.ndarray) -> np.ndarray:
    """
    Run sliding-window inference on a (2, H, W) image.
    Returns sigmoid probability map (H, W) in [0, 1].
    """
    _, H, W = img.shape
    prob_sum   = np.zeros((H, W), dtype=np.float32)
    count_map  = np.zeros((H, W), dtype=np.float32)

    # Collect all tile positions
    ys = list(range(0, H - TILE_SIZE + 1, STRIDE))
    xs = list(range(0, W - TILE_SIZE + 1, STRIDE))
    if not ys or H < TILE_SIZE:
        ys = [0]
    if not xs or W < TILE_SIZE:
        xs = [0]
    # Ensure last row/col is covered
    if ys[-1] + TILE_SIZE < H:
        ys.append(H - TILE_SIZE)
    if xs[-1] + TILE_SIZE < W:
        xs.append(W - TILE_SIZE)

    positions = [(y, x) for y in ys for x in xs]

    tiles, coords = [], []

    def run_batch(tiles, coords):
        t = torch.from_numpy(np.stack(tiles)).to(DEVICE)  # (B, 2, T, T)
        # Pad to TILE_SIZE if needed (edge tiles)
        with torch.amp.autocast('cuda'):
            with torch.no_grad():
                logits = model(t)  # (B, 1, T, T)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()  # (B, T, T)
        for prob, (y, x) in zip(probs, coords):
            ph, pw = prob.shape
            prob_sum[y:y+ph, x:x+pw] += prob
            count_map[y:y+ph, x:x+pw] += 1.0

    for y, x in positions:
        tile = img[:, y:y+TILE_SIZE, x:x+TILE_SIZE]
        # Pad if at boundary
        ph, pw = tile.shape[1], tile.shape[2]
        if ph < TILE_SIZE or pw < TILE_SIZE:
            pad = np.zeros((2, TILE_SIZE, TILE_SIZE), dtype=np.float32)
            pad[:, :ph, :pw] = tile
            tile = pad
        tiles.append(tile)
        coords.append((y, x))
        if len(tiles) == BATCH_TILES:
            run_batch(tiles, coords)
            tiles, coords = [], []
    if tiles:
        run_batch(tiles, coords)

    count_map = np.maximum(count_map, 1.0)
    return prob_sum / count_map


print('✓ Inference helpers defined')

In [ ]:
# ── Run inference on all scenes ────────────────────────────────────────────
# We collect pixel-level gt/pred for threshold selection,
# plus scene-level max/mean prob for scene classification.

all_gt_flat   = []   # pixel gt labels (for threshold PR curve)
all_pred_flat = []   # pixel predicted probs
scene_rows    = []   # per-scene summary

N = len(df)
t0 = time.time()

for i, row in df.iterrows():
    img  = read_scene(row['img_path'])
    mask = read_mask(row['mask_path'])
    prob = inference_scene(img)

    # Flatten for pixel-level analysis
    all_gt_flat.append(mask.ravel())
    all_pred_flat.append(prob.ravel())

    # Scene-level stats
    oil_px     = int(mask.sum())
    scene_max  = float(prob.max())
    scene_mean = float(prob.mean())

    scene_rows.append({
        'scene_id':   row['scene_id'],
        'category':   row['category'],
        'oil_pixels_gt': oil_px,
        'scene_max_prob':  scene_max,
        'scene_mean_prob': scene_mean,
    })

    elapsed = time.time() - t0
    done    = i + 1
    eta     = elapsed / done * (N - done)
    print(f'[{done:3d}/{N}] {row["scene_id"]} | cat={row["category"]:9s} '
          f'| max_p={scene_max:.3f} | oil_px={oil_px:6d} '
          f'| ETA {eta/60:.1f}m', flush=True)

scene_df = pd.DataFrame(scene_rows)
print(f'\n✓ Inference complete in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Threshold selection from pixel-level PR curve ──────────────────────────
print('Computing precision-recall curve …')
gt_arr   = np.concatenate(all_gt_flat).astype(np.int32)
pred_arr = np.concatenate(all_pred_flat).astype(np.float32)

precision, recall, thresholds = precision_recall_curve(gt_arr, pred_arr)
ap = average_precision_score(gt_arr, pred_arr)
print(f'  Average Precision (pixel): {ap:.4f}')

# Best F1 threshold
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)
best_idx  = int(np.argmax(f1_scores))
BEST_THRESHOLD = float(thresholds[best_idx])
best_f1        = float(f1_scores[best_idx])
best_p         = float(precision[best_idx])
best_r         = float(recall[best_idx])

print(f'\n  Best threshold : {BEST_THRESHOLD:.4f}')
print(f'  Best F1        : {best_f1:.4f}')
print(f'  Precision      : {best_p:.4f}')
print(f'  Recall         : {best_r:.4f}')

# ── Plot PR curve ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, lw=1.5, color='steelblue', label=f'AP={ap:.3f}')
ax.scatter([best_r], [best_p], color='crimson', zorder=5,
           label=f'Best F1={best_f1:.3f}\nThresh={BEST_THRESHOLD:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Pixel-level Precision-Recall Curve (Part 3, all 450 scenes)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / 'pr_curve.png', dpi=150)
plt.close()
print('✓ pr_curve.png saved')

In [ ]:
# ── Scene-level classification at the selected threshold ───────────────────
# A scene is flagged as "oil detected" if any pixel prob >= BEST_THRESHOLD
# (i.e. scene_max_prob >= BEST_THRESHOLD).
# Ground truth: oil scenes have oil_pixels_gt > 0.

scene_df['detected']  = (scene_df['scene_max_prob'] >= BEST_THRESHOLD).astype(int)
scene_df['has_oil_gt'] = (scene_df['oil_pixels_gt'] > 0).astype(int)

# Per-category summary
print('\n=== Scene-level detection summary ===')
for cat in ['oil', 'lookalike', 'clean']:
    sub = scene_df[scene_df['category'] == cat]
    n_detected = int(sub['detected'].sum())
    n_total    = len(sub)
    print(f'  {cat:10s}: {n_detected:3d}/{n_total:3d} flagged as oil '
          f'| mean_max_prob={sub["scene_max_prob"].mean():.3f}')

# Overall pixel metrics at threshold
pred_binary = (pred_arr >= BEST_THRESHOLD).astype(np.int32)
tp = int(((pred_binary == 1) & (gt_arr == 1)).sum())
fp = int(((pred_binary == 1) & (gt_arr == 0)).sum())
fn = int(((pred_binary == 0) & (gt_arr == 1)).sum())
tn = int(((pred_binary == 0) & (gt_arr == 0)).sum())

pixel_prec = tp / (tp + fp + 1e-8)
pixel_rec  = tp / (tp + fn + 1e-8)
pixel_iou  = tp / (tp + fp + fn + 1e-8)
pixel_f1   = 2 * pixel_prec * pixel_rec / (pixel_prec + pixel_rec + 1e-8)

print(f'\n=== Pixel metrics @ threshold={BEST_THRESHOLD:.4f} ===')
print(f'  IoU       : {pixel_iou:.4f}')
print(f'  F1 / Dice : {pixel_f1:.4f}')
print(f'  Precision : {pixel_prec:.4f}')
print(f'  Recall    : {pixel_rec:.4f}')

In [ ]:
# ── Confusion matrix (scene-level, 3-class) ────────────────────────────────
# We'll treat Oil scenes that are NOT detected as misses,
# and Lookalike/Clean scenes that ARE detected as false alarms.

# For 3-class confusion we define predicted label:
#   detected & oil_gt=1  → 'oil'
#   detected & oil_gt=0  → 'false_alarm' (use category as pred)
# This is approximate — a full triage classifier would be a separate CNN.
# Here we use rule: predicted=category if not detected, else 'oil'.

y_true = scene_df['category'].values
y_pred = np.where(
    scene_df['detected'].values == 1,
    'oil',          # model says oil
    scene_df['category'].values   # model says no-oil → assign true category
)

labels = ['oil', 'lookalike', 'clean']
cm = confusion_matrix(y_true, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Scene-level Confusion Matrix\n(threshold={BEST_THRESHOLD:.3f})')
plt.tight_layout()
plt.savefig(WORK_DIR / 'confusion_matrix.png', dpi=150)
plt.close()
print('✓ confusion_matrix.png saved')

In [ ]:
# ── Save triage report & scene CSV ────────────────────────────────────────
import datetime

report = {
    'completed_utc': datetime.datetime.utcnow().isoformat(),
    'checkpoint':    str(CKPT_PATH),
    'n_scenes':      len(scene_df),
    'threshold':     BEST_THRESHOLD,
    'average_precision_pixel': round(ap, 4),
    'pixel_metrics_at_threshold': {
        'iou':       round(pixel_iou, 4),
        'f1_dice':   round(pixel_f1,  4),
        'precision': round(pixel_prec, 4),
        'recall':    round(pixel_rec,  4),
    },
    'scene_detection_rate': {
        cat: {
            'detected': int(scene_df[scene_df['category'] == cat]['detected'].sum()),
            'total':    int((scene_df['category'] == cat).sum()),
        }
        for cat in ['oil', 'lookalike', 'clean']
    },
}

with open(WORK_DIR / 'triage_report.json', 'w') as f:
    json.dump(report, f, indent=2)

scene_df.to_csv(WORK_DIR / 'scene_results.csv', index=False)

print('\n=== Triage Report ===')
print(json.dumps(report, indent=2))
print('\n✓ triage_report.json saved')
print('✓ scene_results.csv saved')
print('\nOutputs ready for download:')
import subprocess
subprocess.run(['ls', '-lh',
                str(WORK_DIR / 'triage_report.json'),
                str(WORK_DIR / 'pr_curve.png'),
                str(WORK_DIR / 'confusion_matrix.png'),
                str(WORK_DIR / 'scene_results.csv')])